In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
target_channels = ['NO2', 'C2H5OH', 'VOC', 'CO']
available_substances = ['banana', 'orange', 'pear', 'apple', 'mango', 'peach', 'strawberry', 'cloves', 'coriander', 'garlic', 'almond', 'cumin']

root_dir = "/home/dewei/workspace/SmellNet/four_channel_dataset/Mixtures_Smell_Data"
new_root_dir = "/home/dewei/workspace/SmellNet/four_channel"

os.makedirs(new_root_dir, exist_ok=True)

In [ ]:
for dirpath, dirnames, filenames in os.walk(root_dir):
    for ingredient in dirnames:
        new_path = os.path.join(dirpath, ingredient)
        for sub_dirpath, sub_dirnames, sub_filenames in os.walk(new_path):
            for filename in sub_filenames:
                if filename.endswith(".csv"):  # Process only CSV files
                    file_path = os.path.join(sub_dirpath, filename)
                    
                    # Read and filter dataset
                    df = pd.read_csv(file_path)
                    new_df = df[target_channels]
                    
                    # Build new save path preserving directory structure
                    relative_path = os.path.relpath(sub_dirpath, root_dir)
                    save_dir = os.path.join(new_root_dir, relative_path)
                    os.makedirs(save_dir, exist_ok=True)
                    
                    new_file_path = os.path.join(save_dir, filename)
                    new_df.to_csv(new_file_path, index=False)

Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/almond/almond.1358d2b56280.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/almond/almond.4016eb19a226.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/almond/almond.8efe4fc13fd7.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/almond/almond.a5807139ebbd.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/almond/almond.d31ddd049b10.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/apple/apple.31a9872a75df.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/apple/apple.53398c87211d.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/apple/apple.92a13564a60c.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/apple/apple.b79ec5501b57.csv.csv
Saved filtered file: /home/dewei/workspace/SmellNet/four_channel/apple/apple.c2dd844ef205.csv.csv
Saved filt

In [3]:
def align_datasets(new_df):
    new_df = new_df.copy()
    
    # Take every 10th row (downsampling from 10Hz to 1Hz)
    new_resampled = new_df.iloc[::10].reset_index(drop=True)

    # Keep only target channels
    new_resampled = new_resampled[target_channels]

    return new_resampled


In [4]:
new_data_root = "/home/dewei/workspace/SmellNet/four_channel_test"
new_root_dir = "/home/dewei/workspace/SmellNet/four_channel_real_time_test_3"

os.makedirs(new_root_dir, exist_ok=True)

for dirpath, dirnames, filenames in os.walk(new_data_root):
    for ingredient in dirnames:
        new_path = os.path.join(dirpath, ingredient)
        for sub_dirpath, sub_dirnames, sub_filenames in os.walk(new_path):
            for filename in sub_filenames:
                if filename.endswith(".csv"):  # Process only CSV files
                    file_path = os.path.join(sub_dirpath, filename)
                    
                    # Read and filter dataset
                    df = pd.read_csv(file_path)
                    df.rename(columns={"C2H5CH": "C2H5OH"}, inplace=True)
                    new_df = align_datasets(df[target_channels])
                    
                    # Build new save path preserving directory structure
                    relative_path = os.path.relpath(sub_dirpath, new_data_root)
                    save_dir = os.path.join(new_root_dir, relative_path)
                    os.makedirs(save_dir, exist_ok=True)
                    
                    new_file_path = os.path.join(save_dir, filename)
                    new_df.to_csv(new_file_path, index=False)

In [1]:
import os
import hashlib

def hash_file(filepath):
    """Returns a hash (SHA256) for the contents of a file."""
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def find_matching_files(dir_a, dir_b):
    """Finds matching file contents between dir_a and dir_b."""
    hash_to_files_a = {}
    hash_to_files_b = {}

    # Hash all files in A
    for root, _, files in os.walk(dir_a):
        for fname in files:
            path = os.path.join(root, fname)
            h = hash_file(path)
            hash_to_files_a.setdefault(h, []).append(path)

    # Hash all files in B
    for root, _, files in os.walk(dir_b):
        for fname in files:
            path = os.path.join(root, fname)
            h = hash_file(path)
            hash_to_files_b.setdefault(h, []).append(path)

    # Find matches
    matches = []
    for h in hash_to_files_a:
        if h in hash_to_files_b:
            for f1 in hash_to_files_a[h]:
                for f2 in hash_to_files_b[h]:
                    matches.append((f1, f2))

    return matches


In [2]:
matches = find_matching_files("/home/dewei/workspace/SmellNet/four_channel_real_time_test_2", "/home/dewei/workspace/SmellNet/four_channel")

print("Matching CSV file contents (regardless of filename):")
for a, b in matches:
    print(f"{a}  ==  {b}")

Matching CSV file contents (regardless of filename):
